
# SuperbCommand – Multi-Asset CTA Strategy
## Notebook 00 — Research Configuration & Project Initialisation

**Project inception date:** 5 September 2026  
**Research cutoff:** 31 December 2025  
**Locked holdout:** 1 January 2026 onward

This notebook establishes the global research environment for the SuperbCommand Multi-Asset CTA Strategy project.

It is deliberately limited to infrastructure and configuration. It does **not** download market data, calculate signals, optimise parameters, or run backtests.

### Core responsibilities

- Mount Google Drive.
- Define the canonical project directory.
- Create the project folder structure automatically.
- Define global research and holdout dates.
- Protect the 2026 holdout period from accidental use.
- Define random seeds and reproducibility settings.
- Define standard portfolio, optimisation, and transaction-cost defaults.
- Create reusable helper functions for later notebooks.
- Save the project configuration and a machine-readable manifest to Google Drive.
- Run validation checks so later notebooks can safely depend on this configuration.

### Research principle

> The purpose of the project is not to identify the historically optimal strategy. It is to identify the simplest member of a stable family of strategies with the strongest evidence of future robustness.

The **2026 holdout must remain locked** during model design and optimisation.


In [1]:

# ============================================================
# 0.1 — IMPORTS
# ============================================================

from __future__ import annotations

import json
import os
import platform
import random
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
NumPy: 2.1.3
pandas: 2.2.3


In [2]:

# ============================================================
# 0.2 — MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Google Colab not detected. Drive was not mounted.")

print("Running in Colab:", IN_COLAB)


Mounted at /content/drive
Running in Colab: True



## Canonical project location

All notebooks, data, results, figures, manifests and configuration files for this project will live under:

`My Drive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy`

Later notebooks should import or reproduce this path exactly rather than introducing alternative project roots.


In [3]:

# ============================================================
# 0.3 — PROJECT PATHS
# ============================================================

if IN_COLAB:
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "SuperbCommand/Multi-Asset CTA Strategy"
    )
else:
    # Local fallback for development outside Colab.
    # Change this only if intentionally running the project locally.
    PROJECT_ROOT = Path.cwd() / "Multi-Asset CTA Strategy"

PATHS = {
    "root": PROJECT_ROOT,
    "notebooks": PROJECT_ROOT / "notebooks",
    "data": PROJECT_ROOT / "data",
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "data_cache": PROJECT_ROOT / "data" / "cache",
    "results": PROJECT_ROOT / "results",
    "results_backtests": PROJECT_ROOT / "results" / "backtests",
    "results_optimisation": PROJECT_ROOT / "results" / "optimisation",
    "results_monte_carlo": PROJECT_ROOT / "results" / "monte_carlo",
    "results_holdout": PROJECT_ROOT / "results" / "holdout",
    "figures": PROJECT_ROOT / "figures",
    "manifests": PROJECT_ROOT / "manifests",
    "config": PROJECT_ROOT / "config",
    "logs": PROJECT_ROOT / "logs",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nProject directories:")
for name, path in PATHS.items():
    print(f"  {name:24s} -> {path}")


Project root:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy

Project directories:
  root                     -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy
  notebooks                -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/notebooks
  data                     -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data
  data_raw                 -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/raw
  data_processed           -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/processed
  data_cache               -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/cache
  results                  -> /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/results
  results_backtests        -> /content/drive/MyDrive/Colab Notebooks/Sup


## Holdout policy

The entire 2026 calendar year is reserved as the final holdout.

During strategy development:

- no parameter may be selected using 2026 performance;
- no portfolio design decision may be based on 2026 results;
- no covariance, volatility, signal, or optimisation training window may extend beyond 31 December 2025;
- ordinary research notebooks should fail loudly if they attempt to request holdout observations.

The holdout should only be unlocked in the dedicated final holdout notebook **after the strategy architecture and parameters have been frozen**.


In [4]:

# ============================================================
# 0.4 — GLOBAL RESEARCH DATES & HOLDOUT LOCK
# ============================================================

PROJECT_INCEPTION_DATE = pd.Timestamp("2026-09-05")

# Research data may be used up to and including this date.
RESEARCH_END = pd.Timestamp("2025-12-31")

# Final holdout begins on this date.
HOLDOUT_START = pd.Timestamp("2026-01-01")

# Keep False in every ordinary research notebook.
UNLOCK_HOLDOUT = False

# Earliest date is intentionally left flexible because asset histories differ.
DEFAULT_RESEARCH_START = pd.Timestamp("2005-01-01")

assert RESEARCH_END < HOLDOUT_START
assert PROJECT_INCEPTION_DATE >= HOLDOUT_START

print("Project inception :", PROJECT_INCEPTION_DATE.date())
print("Default start     :", DEFAULT_RESEARCH_START.date())
print("Research end      :", RESEARCH_END.date())
print("Holdout start     :", HOLDOUT_START.date())
print("Holdout unlocked? :", UNLOCK_HOLDOUT)


Project inception : 2026-09-05
Default start     : 2005-01-01
Research end      : 2025-12-31
Holdout start     : 2026-01-01
Holdout unlocked? : False


In [5]:

# ============================================================
# 0.5 — HOLDOUT PROTECTION HELPERS
# ============================================================

class HoldoutAccessError(RuntimeError):
    """Raised when a research process attempts to use locked holdout data."""


def _as_timestamp(value: Any) -> pd.Timestamp:
    """Convert a date-like object into a timezone-naive pandas Timestamp."""
    ts = pd.Timestamp(value)
    if ts.tz is not None:
        ts = ts.tz_convert(None)
    return ts


def assert_research_window(
    start: Any,
    end: Any,
    *,
    unlock_holdout: bool = UNLOCK_HOLDOUT,
) -> None:
    """
    Validate that a requested date range does not enter the locked holdout.

    Parameters
    ----------
    start, end
        Date-like objects.
    unlock_holdout
        Must remain False during ordinary research.
    """
    start_ts = _as_timestamp(start)
    end_ts = _as_timestamp(end)

    if start_ts > end_ts:
        raise ValueError(f"Start date {start_ts.date()} is after end date {end_ts.date()}.")

    if not unlock_holdout and end_ts >= HOLDOUT_START:
        raise HoldoutAccessError(
            "LOCKED HOLDOUT ACCESS BLOCKED. "
            f"Requested end date {end_ts.date()} reaches the holdout beginning "
            f"{HOLDOUT_START.date()}. Ordinary research must stop at "
            f"{RESEARCH_END.date()}."
        )


def clip_to_research_period(
    obj: pd.Series | pd.DataFrame,
    *,
    unlock_holdout: bool = UNLOCK_HOLDOUT,
) -> pd.Series | pd.DataFrame:
    """
    Return an object restricted to the permitted research period.

    This is a convenience function, not a substitute for explicit checks.
    """
    if unlock_holdout:
        return obj.copy()

    if not isinstance(obj.index, pd.DatetimeIndex):
        raise TypeError("Object index must be a pandas DatetimeIndex.")

    result = obj.loc[obj.index <= RESEARCH_END].copy()

    if len(result) == 0:
        raise ValueError("No observations remain after applying the research cutoff.")

    return result


def assert_no_holdout_rows(
    obj: pd.Series | pd.DataFrame,
    *,
    unlock_holdout: bool = UNLOCK_HOLDOUT,
) -> None:
    """Fail if a time-indexed object contains locked holdout observations."""
    if unlock_holdout:
        return

    if not isinstance(obj.index, pd.DatetimeIndex):
        raise TypeError("Object index must be a pandas DatetimeIndex.")

    max_date = obj.index.max()
    if pd.Timestamp(max_date) >= HOLDOUT_START:
        raise HoldoutAccessError(
            f"Object contains data through {pd.Timestamp(max_date).date()}, "
            f"but the research cutoff is {RESEARCH_END.date()}."
        )


# Demonstration: permitted research request
assert_research_window("2010-01-01", "2025-12-31")
print("Research-window guard: PASS")

# Demonstration: intentionally verify that the holdout is blocked
try:
    assert_research_window("2010-01-01", "2026-01-02")
except HoldoutAccessError:
    print("Holdout lock test: PASS")
else:
    raise AssertionError("Holdout protection failed.")


Research-window guard: PASS
Holdout lock test: PASS



## Reproducibility

All stochastic research—random search, Monte Carlo simulations, bootstrap procedures, universe resampling and parameter perturbation—should derive from explicit seeds.

Later notebooks may use additional child seeds, but the project-level seed below is the canonical starting point.


In [6]:

# ============================================================
# 0.6 — REPRODUCIBILITY
# ============================================================

GLOBAL_RANDOM_SEED = 20260905

random.seed(GLOBAL_RANDOM_SEED)
np.random.seed(GLOBAL_RANDOM_SEED)

RNG = np.random.default_rng(GLOBAL_RANDOM_SEED)

print("Global random seed:", GLOBAL_RANDOM_SEED)


Global random seed: 20260905



## Standard research defaults

These are **initial defaults, not optimised conclusions**.

Their purpose is to ensure that every notebook starts from one coherent set of assumptions. Later research may compare alternatives, but any departure should be explicit and recorded.

The defaults deliberately favour conservative, interpretable choices.


In [7]:

# ============================================================
# 0.7 — STANDARD RESEARCH DEFAULTS
# ============================================================

@dataclass(frozen=True)
class ResearchDefaults:
    # Data
    trading_days_per_year: int = 252
    price_field: str = "Adj Close"
    base_currency: str = "USD"

    # Return calculations
    return_type: str = "simple"
    annualisation_factor: int = 252

    # Volatility estimation
    vol_lookback_days: int = 63          # ~3 months
    vol_floor_annual: float = 0.03       # avoid pathological sizing
    vol_cap_annual: float = 1.50

    # Covariance estimation
    covariance_lookback_days: int = 252
    covariance_min_obs: int = 126

    # Portfolio
    portfolio_vol_target: float = 0.10
    max_gross_leverage: float = 2.00
    max_single_market_risk_weight: float = 0.20

    # Rebalancing
    default_rebalance_frequency: str = "W-FRI"

    # Transaction-cost placeholders
    # These are intentionally simple during the ETF prototype phase.
    transaction_cost_bps_one_way: float = 2.0
    slippage_bps_one_way: float = 1.0

    # Optimisation
    walk_forward_train_years: int = 8
    walk_forward_test_years: int = 2
    minimum_train_years: int = 5

    # Monte Carlo / robustness
    monte_carlo_iterations: int = 2_000
    bootstrap_block_days: int = 20
    parameter_perturbation_pct: float = 0.10
    universe_dropout_pct: float = 0.20

    # Guardrail
    research_end: str = "2025-12-31"
    holdout_start: str = "2026-01-01"


DEFAULTS = ResearchDefaults()

for key, value in asdict(DEFAULTS).items():
    print(f"{key:32s}: {value}")


trading_days_per_year           : 252
price_field                     : Adj Close
base_currency                   : USD
return_type                     : simple
annualisation_factor            : 252
vol_lookback_days               : 63
vol_floor_annual                : 0.03
vol_cap_annual                  : 1.5
covariance_lookback_days        : 252
covariance_min_obs              : 126
portfolio_vol_target            : 0.1
max_gross_leverage              : 2.0
max_single_market_risk_weight   : 0.2
default_rebalance_frequency     : W-FRI
transaction_cost_bps_one_way    : 2.0
slippage_bps_one_way            : 1.0
walk_forward_train_years        : 8
walk_forward_test_years         : 2
minimum_train_years             : 5
monte_carlo_iterations          : 2000
bootstrap_block_days            : 20
parameter_perturbation_pct      : 0.1
universe_dropout_pct            : 0.2
research_end                    : 2025-12-31
holdout_start                   : 2026-01-01



## Initial research universe philosophy

Notebook 00 does **not** define the final investable universe.

Notebook 01 will construct the actual prototype universe. The guiding architecture is:

- Equity indices
- Government rates
- Commodities
- FX
- Crypto

During the rapid-prototyping phase, liquid ETF/index proxies may be used where this materially simplifies data acquisition. The final architecture is expected to migrate toward liquid futures where practical.


In [8]:

# ============================================================
# 0.8 — CANONICAL ASSET-CLASS LABELS
# ============================================================

ASSET_CLASSES = (
    "equities",
    "rates",
    "commodities",
    "fx",
    "crypto",
)

PORTFOLIO_METHODS = (
    "equal_weight",
    "inverse_volatility",
    "equal_risk_contribution",
    "erc_vol_targeted",
)

STRATEGY_DIRECTIONS = (
    "long_cash",
    "long_short",
)

print("Asset classes      :", ASSET_CLASSES)
print("Portfolio methods  :", PORTFOLIO_METHODS)
print("Strategy directions:", STRATEGY_DIRECTIONS)


Asset classes      : ('equities', 'rates', 'commodities', 'fx', 'crypto')
Portfolio methods  : ('equal_weight', 'inverse_volatility', 'equal_risk_contribution', 'erc_vol_targeted')
Strategy directions: ('long_cash', 'long_short')



## Research status and configuration files

This notebook writes machine-readable configuration files into the project directory.

Later notebooks should load these files rather than relying on manually retyped assumptions wherever practical.

This provides an audit trail and reduces accidental drift between notebooks.


In [9]:

# ============================================================
# 0.9 — SAVE PROJECT CONFIGURATION
# ============================================================

CONFIG = {
    "project": {
        "name": "SuperbCommand – Multi-Asset CTA Strategy",
        "notebook": "00 — Research Configuration & Project Initialisation",
        "project_inception_date": str(PROJECT_INCEPTION_DATE.date()),
        "project_root": str(PROJECT_ROOT),
    },
    "research_dates": {
        "default_research_start": str(DEFAULT_RESEARCH_START.date()),
        "research_end": str(RESEARCH_END.date()),
        "holdout_start": str(HOLDOUT_START.date()),
        "unlock_holdout": UNLOCK_HOLDOUT,
    },
    "reproducibility": {
        "global_random_seed": GLOBAL_RANDOM_SEED,
    },
    "defaults": asdict(DEFAULTS),
    "asset_classes": list(ASSET_CLASSES),
    "portfolio_methods": list(PORTFOLIO_METHODS),
    "strategy_directions": list(STRATEGY_DIRECTIONS),
    "paths": {name: str(path) for name, path in PATHS.items()},
}

config_path = PATHS["config"] / "project_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print("Saved configuration:")
print(config_path)


Saved configuration:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/config/project_config.json


In [10]:

# ============================================================
# 0.10 — WRITE NOTEBOOK MANIFEST
# ============================================================

run_timestamp_utc = datetime.now(timezone.utc).isoformat()

MANIFEST = {
    "notebook_id": "00",
    "notebook_name": "Research Configuration & Project Initialisation",
    "project_name": "SuperbCommand – Multi-Asset CTA Strategy",
    "run_timestamp_utc": run_timestamp_utc,
    "python_version": sys.version,
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "research_end": str(RESEARCH_END.date()),
    "holdout_start": str(HOLDOUT_START.date()),
    "holdout_unlocked": UNLOCK_HOLDOUT,
    "global_random_seed": GLOBAL_RANDOM_SEED,
    "status": "INITIALISED",
}

manifest_path = PATHS["manifests"] / "00_research_configuration_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)

print("Saved manifest:")
print(manifest_path)


Saved manifest:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/manifests/00_research_configuration_manifest.json



## Validation suite

The final cell performs a small set of checks before later notebooks are allowed to depend on this configuration.

A successful run should end with:

`NOTEBOOK 00 STATUS: PASS`


In [11]:

# ============================================================
# 0.11 — VALIDATION SUITE
# ============================================================

checks = {}

# 1. Project root exists
checks["project_root_exists"] = PROJECT_ROOT.exists()

# 2. Every required directory exists
checks["all_directories_exist"] = all(path.exists() for path in PATHS.values())

# 3. Config file exists
checks["config_written"] = config_path.exists()

# 4. Manifest exists
checks["manifest_written"] = manifest_path.exists()

# 5. Research / holdout dates are correctly ordered
checks["date_order_valid"] = RESEARCH_END < HOLDOUT_START <= PROJECT_INCEPTION_DATE

# 6. Holdout is locked
checks["holdout_locked"] = (UNLOCK_HOLDOUT is False)

# 7. Guard blocks holdout access
try:
    assert_research_window("2020-01-01", "2026-01-01")
except HoldoutAccessError:
    checks["holdout_guard_operational"] = True
else:
    checks["holdout_guard_operational"] = False

# 8. Random generator works reproducibly
test_rng_1 = np.random.default_rng(GLOBAL_RANDOM_SEED).normal(size=5)
test_rng_2 = np.random.default_rng(GLOBAL_RANDOM_SEED).normal(size=5)
checks["rng_reproducible"] = np.allclose(test_rng_1, test_rng_2)

validation_df = pd.DataFrame(
    {"check": list(checks.keys()), "passed": list(checks.values())}
)

print(validation_df.to_string(index=False))

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(f"Notebook 00 validation failed: {failed}")

print("\n" + "=" * 72)
print("NOTEBOOK 00 STATUS: PASS")
print("=" * 72)
print("Project infrastructure is initialised.")
print(f"Research data cutoff remains locked at {RESEARCH_END.date()}.")
print("Proceed to Notebook 01 — Universe & Data Engine.")


                    check  passed
      project_root_exists    True
    all_directories_exist    True
           config_written    True
         manifest_written    True
         date_order_valid    True
           holdout_locked    True
holdout_guard_operational    True
         rng_reproducible    True

NOTEBOOK 00 STATUS: PASS
Project infrastructure is initialised.
Research data cutoff remains locked at 2025-12-31.
Proceed to Notebook 01 — Universe & Data Engine.



---

## End of Notebook 00

### Outputs created

- Project directory structure
- `config/project_config.json`
- `manifests/00_research_configuration_manifest.json`

### Next notebook

**Notebook 01 — Universe & Data Engine**

Notebook 01 should:

1. define the prototype cross-asset universe;
2. obtain price data from the initial research source;
3. standardise and validate the data;
4. cache raw and processed datasets;
5. prevent post-2025 data from entering ordinary research;
6. produce a concise data-quality report before any signal calculations begin.
